# Process data reconciliation - Exercise 1

This notebook solves the exercise with two objective functions:

1. Mean squared error in **absolute flowrate** units.
2. Mean squared **relative** error.

The code is deliberately sequential: it contains no user-defined functions.

## Measurements and process constraints

The process must satisfy the following four steady-state mass balances:

$$f_1+f_7-f_2=0 \qquad \text{Mixer}$$
$$f_2-f_3=0 \qquad \text{Reactor}$$
$$f_3-f_4-f_5=0 \qquad \text{Column}$$
$$f_5-f_6-f_7=0 \qquad \text{Splitter}$$

The measurements do not satisfy these balances exactly because of measurement error.

In [22]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

#pd.set_option('display.float_format', '{:,.3f}'.format)

# Measured flowrates [kg/h], ordered as f1, f2, ..., f7
stream_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
f_meas = np.array([95.0, 170.0, 175.0, 75.0, 103.0, 15.0, 82.0])

# Standard deviations of the measurements [kg/h].
# They are not given in this exercise, so NaN means 'not available', not zero.
std_meas = np.array([np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])

input_table = pd.DataFrame({
    'Stream': stream_names,
    'Measured flowrate [kg/h]': f_meas,
    'Standard deviation [kg/h]': std_meas,
})

input_table

,Stream,Measured flowrate [kg/h],Standard deviation [kg/h]
0,f1,95.000,NaN
1,f2,170.000,NaN
2,f3,175.000,NaN
3,f4,75.000,NaN
4,f5,103.000,NaN
5,f6,15.000,NaN
6,f7,82.000,NaN


If reliable standard deviations were available, the statistically preferred objective would be:

$$\sum_i\left(\frac{f_i^{R}-f_i^{m}}{\sigma_i}\right)^2$$

Because no standard deviations are provided here, we compare absolute and relative errors instead.

In [23]:
# Check the four balances using the original measurements.
# A zero value would mean that the corresponding balance is satisfied.
measured_balance_residuals = np.array([
    f_meas[0] + f_meas[6] - f_meas[1],  # f1 + f7 - f2
    f_meas[1] - f_meas[2],              # f2 - f3
    f_meas[2] - f_meas[3] - f_meas[4],  # f3 - f4 - f5
    f_meas[4] - f_meas[5] - f_meas[6],  # f5 - f6 - f7
])

balance_table = pd.DataFrame({
    'Unit': ['Mixer', 'Reactor', 'Column', 'Splitter'],
    'Balance residual using measurements [kg/h]': measured_balance_residuals,
})

balance_table

,Unit,Balance residual using measurements [kg/h]
0,Mixer,7.000
1,Reactor,-5.000
2,Column,-3.000
3,Splitter,6.000


## Solution 1 - Absolute mean squared error

The objective function is:

$$\text{MSE}=\frac{1}{7}\sum_{i=1}^{7}(f_i^R-f_i^m)^2$$

Large and small flowrates are treated equally in kg/h.

In [24]:
# The four equality constraints are written explicitly.
# Each lambda expression receives the current trial values of f1, ..., f7.
constraints = [
    {'type': 'eq', 'fun': lambda f: f[0] + f[6] - f[1]},
    {'type': 'eq', 'fun': lambda f: f[1] - f[2]},
    {'type': 'eq', 'fun': lambda f: f[2] - f[3] - f[4]},
    {'type': 'eq', 'fun': lambda f: f[4] - f[5] - f[6]},
]

# Starting point = original measurements.
# SLSQP changes the flowrates as little as possible while enforcing all balances.
absolute_solution = minimize(
    lambda f: np.mean((f - f_meas)**2),
    x0=f_meas,
    constraints=constraints,
    method='SLSQP',
    options={'ftol': 1e-12, 'maxiter': 1000},
)

if not absolute_solution.success:
    raise RuntimeError(absolute_solution.message)

f_rec_absolute = absolute_solution.x
absolute_error = f_rec_absolute - f_meas
absolute_error_pct = 100 * absolute_error / f_meas

absolute_results = pd.DataFrame({
    'Stream': stream_names,
    'Measured [kg/h]': f_meas,
    'Reconciled [kg/h]': f_rec_absolute,
    'Correction [kg/h]': absolute_error,
    'Correction [%]': absolute_error_pct,
})

absolute_results

,Stream,Measured [kg/h],Reconciled [kg/h],Correction [kg/h],Correction [%]
0,f1,95.000,92.208,-2.792,-2.939
1,f2,170.000,174.167,4.167,2.451
2,f3,175.000,174.167,-0.833,-0.476
3,f4,75.000,74.458,-0.542,-0.722
4,f5,103.000,99.708,-3.292,-3.196
5,f6,15.000,17.750,2.750,18.333
6,f7,82.000,81.958,-0.042,-0.051


In [25]:
absolute_balance_check = pd.DataFrame({
    'Unit': ['Mixer', 'Reactor', 'Column', 'Splitter'],
    'Balance residual after reconciliation [kg/h]': [
        f_rec_absolute[0] + f_rec_absolute[6] - f_rec_absolute[1],
        f_rec_absolute[1] - f_rec_absolute[2],
        f_rec_absolute[2] - f_rec_absolute[3] - f_rec_absolute[4],
        f_rec_absolute[4] - f_rec_absolute[5] - f_rec_absolute[6],
    ],
})

absolute_balance_check

,Unit,Balance residual after reconciliation [kg/h]
0,Mixer,0.000
1,Reactor,0.000
2,Column,0.000
3,Splitter,0.000


## Solution 2 - Relative mean squared error

The objective function is:

$$\text{Relative MSE}=\frac{1}{7}\sum_{i=1}^{7}\left(\frac{f_i^R-f_i^m}{f_i^m}\right)^2$$

A 10% correction has the same importance for every stream, regardless of its flowrate.

In [26]:
relative_solution = minimize(
    lambda f: np.mean(((f - f_meas) / f_meas)**2),
    x0=f_meas,
    constraints=constraints,
    method='SLSQP',
    options={'ftol': 1e-12, 'maxiter': 1000},
)

if not relative_solution.success:
    raise RuntimeError(relative_solution.message)

f_rec_relative = relative_solution.x
relative_error = f_rec_relative - f_meas
relative_error_pct = 100 * relative_error / f_meas

relative_results = pd.DataFrame({
    'Stream': stream_names,
    'Measured [kg/h]': f_meas,
    'Reconciled [kg/h]': f_rec_relative,
    'Correction [kg/h]': relative_error,
    'Correction [%]': relative_error_pct,
})

relative_results

,Stream,Measured [kg/h],Reconciled [kg/h],Correction [kg/h],Correction [%]
0,f1,95.000,91.413,-3.587,-3.776
1,f2,170.000,174.979,4.979,2.929
2,f3,175.000,174.979,-0.021,-0.012
3,f4,75.000,76.271,1.271,1.694
4,f5,103.000,98.708,-4.292,-4.167
5,f6,15.000,15.142,0.142,0.946
6,f7,82.000,83.566,1.566,1.910


In [27]:
relative_balance_check = pd.DataFrame({
    'Unit': ['Mixer', 'Reactor', 'Column', 'Splitter'],
    'Balance residual after reconciliation [kg/h]': [
        f_rec_relative[0] + f_rec_relative[6] - f_rec_relative[1],
        f_rec_relative[1] - f_rec_relative[2],
        f_rec_relative[2] - f_rec_relative[3] - f_rec_relative[4],
        f_rec_relative[4] - f_rec_relative[5] - f_rec_relative[6],
    ],
})

relative_balance_check

,Unit,Balance residual after reconciliation [kg/h]
0,Mixer,0.000
1,Reactor,0.000
2,Column,0.000
3,Splitter,-0.000


## Comparison

In [28]:
comparison = pd.DataFrame({
    'Stream': stream_names,
    'Measured [kg/h]': f_meas,
    'Absolute MSE reconciled [kg/h]': f_rec_absolute,
    'Relative MSE reconciled [kg/h]': f_rec_relative,
})

comparison

,Stream,Measured [kg/h],Absolute MSE reconciled [kg/h],Relative MSE reconciled [kg/h]
0,f1,95.000,92.208,91.413
1,f2,170.000,174.167,174.979
2,f3,175.000,174.167,174.979
3,f4,75.000,74.458,76.271
4,f5,103.000,99.708,98.708
5,f6,15.000,17.750,15.142
6,f7,82.000,81.958,83.566
